# DataMate trên Colab Pro: toàn bộ pipeline với Qwen local

Chạy lần lượt từ trên xuống: clone repo → Qwen tạo manifest → ingestion toàn bộ train/public test → sao lưu Drive → kiểm tra DuckDB → Qwen rewrite/sinh SQL → chấm train → xuất bài nộp.

Đầu vào là `dataset/train` và `dataset/public_test` trong repo clone. Không cần Gemini API key. Model được nạp một lần và dùng lại giữa các bước. Ingestion dùng Python/DuckDB, mọi tác vụ LLM dùng Qwen local. Manifest được sinh theo từng bảng nguồn, kiểm định schema, cột và bằng chứng; lỗi sẽ dừng bước chuẩn bị dữ liệu.

Chọn GPU trong **Runtime → Change runtime type**. `hf_quant="auto"` chọn lượng tử hóa theo VRAM; mức context vẫn cần kiểm tra thực tế. Kết quả mặc định ở `/content/datamate_sql_out`, bị mất khi Colab xóa runtime. Ô 3.1 sao lưu dữ liệu ingestion sang Drive để giữ qua các phiên; `OUT` vẫn ở ổ Colab để truy vấn nhanh. Có thể đặt `OUT` trong Drive nếu muốn; khi đó notebook tự mount Drive.

**Chọn dữ liệu ở mục 2:** `DATASET_SPLIT = "train"` (mặc định) chỉ chuẩn bị và đánh giá train. Đổi thành `"public_test"` rồi chạy lại từ mục 2 để chuẩn bị và dự đoán public test. Sao lưu và validation đều theo cờ này.


In [1]:
from dataclasses import dataclass
from pathlib import Path
import base64
import getpass
import os
import subprocess
import sys

@dataclass
class ColabRuntime:
    repo_url: str = "https://github.com/Oszumeiii/ZAIC_2026.git"
    branch: str = "main"
    checkout_dir: str = "/content/ZAIC_2026"
    log_dir: str = "/content/datamate_runtime_logs"

    def checkout(self):
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token cho repo private: ")
        if not token:
            raise RuntimeError("Thiếu GITHUB_TOKEN")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]
        target = Path(self.checkout_dir)
        def run(args, cwd=None):
            result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
            output = (result.stdout + result.stderr).replace(token, "***").replace(auth, "***")
            print(output[-1500:])
            if result.returncode:
                raise RuntimeError("Git lỗi: kiểm tra token, nhánh và thay đổi trong checkout")
        if (target / ".git").is_dir():
            run(git + ["fetch", "origin", self.branch], target)
            run(["git", "checkout", self.branch], target)
            run(git + ["pull", "--ff-only", "origin", self.branch], target)
        else:
            run(git + ["clone", "--branch", self.branch, self.repo_url, str(target)])
        if not (target / "app/local_pipeline.py").is_file():
            raise RuntimeError("Nhánh clone chưa có app/local_pipeline.py; cần push bản full pipeline trước")
        self.root = target.resolve()
        os.chdir(self.root)
        if str(self.root) not in sys.path:
            sys.path.insert(0, str(self.root))
        commit = subprocess.run(["git", "log", "-1", "--oneline"], cwd=self.root,
                                check=True, capture_output=True, text=True)
        print(commit.stdout.strip())
        return self.root

    def run_logged(self, args, *, label, log_name, check=True, env=None):
        """Forward child stdout/stderr into the notebook and save the complete log."""
        log_path = Path(self.log_dir) / log_name
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"▶ {label} | log: {log_path}", flush=True)
        child_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
        with log_path.open("w", encoding="utf-8") as log:
            with subprocess.Popen(
                args, cwd=self.root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding="utf-8", errors="replace", bufsize=1, env=child_env,
            ) as process:
                try:
                    for line in process.stdout:
                        print(line, end="", flush=True)
                        log.write(line)
                        log.flush()
                    returncode = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                    raise
        if check and returncode:
            raise RuntimeError(f"{label} thất bại (exit={returncode}). Xem lỗi phía trên hoặc {log_path}")
        return returncode

    def install(self):
        self.run_logged(
            [sys.executable, "-m", "pip", "install", "-r", str(self.root / "requirements-colab.txt")],
            label="Cài thư viện", log_name="install.log")
        result = self.run_logged(
            [sys.executable, "-m", "pip", "install", "flash-linear-attention"],
            label="Cài kernel tùy chọn", log_name="flash-linear-attention.log", check=False)
        if result:
            print("flash-linear-attention không khả dụng; dùng implementation torch")

    def check_gpu(self):
        self.run_logged(["nvidia-smi"], label="Kiểm tra GPU", log_name="gpu.log")
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Cần runtime có CUDA GPU")
        print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))

    def verify_qwen_device(self, pipeline):
        """Load the shared Qwen backend once and confirm its weights are on CUDA."""
        import torch
        backend = pipeline.llm.backend
        if getattr(backend, "engine", None) != "hf":
            raise RuntimeError(f"Notebook cần backend HF local, nhận {type(backend).__name__}")
        try:
            parameter = next(backend.model.parameters())
        except StopIteration as exc:
            raise RuntimeError("Qwen không có tham số để kiểm tra CUDA") from exc
        device = parameter.device
        if device.type != "cuda":
            raise RuntimeError(f"Qwen đang nằm trên {device}, không phải CUDA")
        index = device.index if device.index is not None else torch.cuda.current_device()
        name = torch.cuda.get_device_name(index)
        print(f"Qwen: {pipeline.config.model} | device={device} | GPU={name} "
              f"| dtype={parameter.dtype} | quant={backend.quant}", flush=True)
        if "A100" not in name.upper():
            print("Runtime hiện tại không báo GPU A100; kiểm tra lại GPU Colab nếu bạn cần A100.", flush=True)

    def verify_manifest_runtime(self):
        """Reject an old checkout or stale imports before loading Qwen/ingesting data."""
        import ast
        import importlib
        source = self.root / "app/manifests/local.py"
        tree = ast.parse(source.read_text(encoding="utf-8"))
        definition = next(node for node in tree.body
                          if isinstance(node, ast.ClassDef) and node.name == "LocalManifestGenerator")
        disk_version = next(ast.literal_eval(node.value) for node in definition.body
                            if isinstance(node, ast.Assign)
                            and any(isinstance(target, ast.Name) and target.id == "VERSION" for target in node.targets))
        has_fix = any(isinstance(node, ast.FunctionDef) and node.name == "canonicalize_metadata"
                      for node in definition.body)
        print(f"Manifest trên đĩa: {disk_version} | {source}", flush=True)
        if not has_fix:
            raise RuntimeError(
                "Repo clone chưa có bản sửa manifest. Commit/push app/manifests/local.py và "
                "app/manifests/validation.py từ máy local, rồi chạy lại ô clone trên Colab.")
        module = importlib.import_module("app.manifests.local")
        orchestration = importlib.import_module("app.local_pipeline")
        loaded = module.LocalManifestGenerator
        bound = orchestration.DatasetPreparationPipeline.prepare.__globals__["LocalManifestGenerator"]
        print(f"Manifest trong RAM: {loaded.VERSION} | pipeline dùng: {bound.VERSION}", flush=True)
        if (Path(module.__file__).resolve() != source.resolve()
                or any(getattr(cls, "VERSION", None) != disk_version
                       or not hasattr(cls, "canonicalize_metadata") for cls in (loaded, bound))):
            raise RuntimeError(
                "File đã cập nhật nhưng Python vẫn giữ class manifest cũ. "
                "Chọn Runtime > Restart session rồi chạy lại các ô từ đầu; không cần xóa OUT.")
        print("Manifest metadata fix: OK", flush=True)

    def run_checks(self):
        self.verify_manifest_runtime()
        import importlib.metadata
        import json
        import platform
        import xml.etree.ElementTree as ET
        logs = Path(self.log_dir)
        logs.mkdir(parents=True, exist_ok=True)
        import hashlib
        revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=self.root,
                                  capture_output=True, text=True, check=True).stdout.strip()
        sql_source = self.root / "app/tools/sql.py"
        versions = {"python": sys.version, "platform": platform.platform(),
                    "git_commit": revision, "sql_executor": str(sql_source),
                    "sql_sha256": hashlib.sha256(sql_source.read_bytes()).hexdigest()}
        for name in ("pytest", "pandas", "numpy", "duckdb", "jsonschema", "openpyxl",
                     "transformers", "torch", "google-genai"):
            try:
                versions[name] = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                versions[name] = "NOT INSTALLED"
        (logs / "environment.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
        print(json.dumps(versions, indent=2), flush=True)
        # Probe the actual checkout in a fresh process, independent of cached notebook imports.
        probe = "\n".join([
            "import duckdb",
            "from app.tools.sql import SQLExecutor",
            "with duckdb.connect() as connection:",
            "    try:",
            "        SQLExecutor.query(connection, 'PRAGMA version')",
            "    except ValueError:",
            "        print('SQL guard: OK — PRAGMA bị chặn')",
            "    else:",
            "        raise SystemExit('Checkout chưa có bản sửa PRAGMA. Commit/push app/tools/sql.py từ máy local, rồi chạy lại ô clone trên Colab.')",
        ])
        self.run_logged([sys.executable, "-c", probe], label="Xác nhận bản sửa SQL",
                        log_name="sql_guard.log")
        report = logs / "pytest.xml"
        report.unlink(missing_ok=True)
        tests = ["tests/test_local_pipeline.py", "tests/test_sql_pipeline.py",
                 "tests/test_architecture.py", "tests/test_business_manifest.py",
                 "tests/test_manifest_v2.py", "tests/test_main.py"]
        result = self.run_logged(
            [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", "--color=no",
             f"--junitxml={report}", *tests],
            label="Kiểm tra pipeline local/SQL", log_name="pytest.log", check=False,
            env={"PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"})
        if result:
            if report.is_file():
                for case in ET.parse(report).iter("testcase"):
                    for kind in ("failure", "error"):
                        issue = case.find(kind)
                        if issue is not None:
                            print(f"{kind.upper()}: {case.get('classname')}.{case.get('name')}", flush=True)
                            print((issue.text or issue.get("message", ""))[-2000:], flush=True)
            raise RuntimeError(
                f"Pytest chưa đạt (exit={result}). Chi tiết: {logs / 'pytest.log'}; "
                f"phiên bản thư viện: {logs / 'environment.json'}. Pipeline chưa được khởi tạo.")
        print("Kiểm tra pipeline local/SQL: OK", flush=True)

    @staticmethod
    def prepare_output(out):
        if str(out).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        Path(out).mkdir(parents=True, exist_ok=True)

## 1. Clone và cài thư viện
Đặt `GITHUB_TOKEN` trong Colab Secrets, bật Notebook access. Nếu trước đó đã import transformers rồi đổi phiên bản thư viện, restart session và chạy lại từ đầu.

In [2]:
runtime = ColabRuntime()
ROOT = runtime.checkout()
runtime.install()
runtime.check_gpu()

Cloning into '/content/ZAIC_2026'...
Updating files:  59% (266/445)
Updating files:  60% (267/445)
Updating files:  61% (272/445)
Updating files:  62% (276/445)
Updating files:  63% (281/445)
Updating files:  64% (285/445)
Updating files:  65% (290/445)
Updating files:  66% (294/445)
Updating files:  67% (299/445)
Updating files:  68% (303/445)
Updating files:  69% (308/445)
Updating files:  70% (312/445)
Updating files:  71% (316/445)
Updating files:  72% (321/445)
Updating files:  73% (325/445)
Updating files:  74% (330/445)
Updating files:  75% (334/445)
Updating files:  76% (339/445)
Updating files:  77% (343/445)
Updating files:  78% (348/445)
Updating files:  79% (352/445)
Updating files:  80% (356/445)
Updating files:  81% (361/445)
Updating files:  82% (365/445)
Updating files:  83% (370/445)
Updating files:  84% (374/445)
Updating files:  85% (379/445)
Updating files:  86% (383/445)
Updating files:  87% (388/445)
Updating files:  88% (392/445)
Updating files:  89% (397/445)
Up

## 2. Kiểm tra mã và cấu hình pipeline
Các test tích hợp SQL/local, manifest và CLI dùng dữ liệu/backend giả, không tải Qwen và không gọi Gemini. Đây là gate cho pipeline này; bộ test toàn repo còn lỗi có sẵn được ghi trong tài liệu. `OUT` giữ manifest, database, báo cáo và cache. Khi chạy lại, manifest chỉ tái sử dụng nếu khớp nguồn, schema và model; DuckDB chỉ tái sử dụng nếu fingerprint khớp.
Output của pytest được in trực tiếp vào ô và lưu tại `/content/datamate_runtime_logs/pytest.log`; phiên bản thư viện nằm trong `environment.json`. Nếu kiểm tra lỗi, notebook dừng và hiển thị tên test thất bại. Không bỏ qua kiểm tra để chạy tiếp.

Lỗi `PRAGMA version` trên DuckDB 1.5.6 đã được sửa ở `app/tools/sql.py`: kiểm tra từ khóa câu SQL gốc trước khi DuckDB phân tích. Sau khi cập nhật mã lên GitHub, chạy lại ô clone để lấy bản sửa rồi chạy ô kiểm tra này; giữ nguyên các test.


In [ ]:
runtime.run_checks()

from app.local_pipeline import LocalPipelineConfig, LocalSQLPipeline

DATASET_SPLIT = "train"  # Đổi thành "public_test" để chạy dự đoán test
if DATASET_SPLIT not in ("train", "public_test"):
    raise ValueError("DATASET_SPLIT phải là train hoặc public_test")
SPLITS = (DATASET_SPLIT,)

OUT = "/content/datamate_sql_out"
# Muốn giữ kết quả qua phiên: OUT = "/content/drive/MyDrive/datamate_sql_out"
runtime.prepare_output(OUT)
config = LocalPipelineConfig(
    dataset_dir=str(ROOT / "dataset"),
    out_dir=OUT,
    model="Qwen/Qwen3.5-9B",
    hf_quant="auto",
    max_model_len=16384,
    manifest_tokens=6144,
    sql_tokens=2048,
    n_samples=3,
    agree_min=2,
)
pipeline = LocalSQLPipeline(config)

## 3. Chuẩn bị dữ liệu của split đã chọn trước agent
Bước này nạp Qwen lần đầu, tạo BusinessManifest v2 cho từng business, kiểm định, rồi dựng DuckDB với Raw/Typed/Semantic và catalog. Manifest đầu ra ở `OUT/manifests/<split>/`, database ở `OUT/databases/<split>/<business>/current/data.duckdb`. Không ghi manifest vào dataset gốc.

Chạy lại ô này để tiếp tục sau gián đoạn; các batch manifest hợp lệ đã lưu được dùng lại. Không chuyển sang agent khi bước này còn báo lỗi. Nếu prompt manifest vượt context, thông báo sẽ chỉ rõ; tăng `max_model_len` trong giới hạn VRAM và khởi tạo lại pipeline sau khi restart session.
Manifest generator `source-batches-v3` chuẩn hóa metadata phụ thuộc: `order_date → time_role=order_date`, `line_amount/base_quantity → aggregation=sum` khi model trả giá trị mâu thuẫn. Giá trị unknown/null được giữ nguyên; kiểm tra nguồn, bằng chứng và đơn vị tính vẫn bắt buộc. Mọi sửa đổi được ghi tại `OUT/manifests/<split>/reports/normalization/<business>/`. Lỗi cuối cùng được lưu tại `reports/failures/`.

Sau khi cập nhật mã từ GitHub, dùng **Runtime → Restart session** rồi chạy lại các ô từ đầu để tránh dùng class Python cũ đã nạp vào RAM. Không cần xóa dataset hoặc thư mục OUT.


In [ ]:
runtime.verify_manifest_runtime()
runtime.verify_qwen_device(pipeline)
preparation = pipeline.prepare_datasets(splits=SPLITS)


### 3.1. Lưu dữ liệu các business sau Data Ingestion
Chạy ô dưới ngay sau khi `prepare_datasets()` hoàn tất. Nếu đã ingestion xong trong phiên hiện tại, chỉ cần chạy ô sao lưu này, không cần chạy lại Qwen. Colab sẽ yêu cầu kết nối Drive ở lần mount đầu tiên.

Mỗi lần chạy tạo một file ZIP mới trong `MyDrive/datamate_ingestion_backups`, giữ nguyên cấu trúc thư mục của `OUT`: DuckDB chứa các lớp Raw/Typed/Semantic và metadata của từng business thuộc `DATASET_SPLIT`; catalog, manifest, tài liệu và báo cáo ingestion; calendar dùng chung và fingerprint; toàn bộ manifest cache/evidence để tiếp tục pipeline. `backup_index.json` liệt kê business và đường dẫn database. Không chứa model Qwen hoặc dataset gốc.

ZIP được tạo trên ổ Colab rồi chép sang Drive; tên `.zip` chỉ xuất hiện khi chép xong. Chờ thông báo **Đã lưu** trước khi ngắt phiên. Dừng các tác vụ ghi database trong lúc sao lưu. Có thể đổi `BACKUP_DIR` sang thư mục local và tải ZIP về máy bằng lệnh cuối ô.

Để dùng lại ở phiên sau: clone cùng phiên bản repo, cài thư viện, giải nén ZIP vào một thư mục `OUT` trống và đặt `config.out_dir` trỏ tới đó. Giữ cùng đường dẫn checkout/dataset vì fingerprint có chứa đường dẫn nguồn. Chạy `pipeline.validate_databases(splits=SPLITS)` trước agent; chỉ chạy lại `prepare_datasets()` nếu dữ liệu/mã ingestion đã thay đổi hoặc thiếu artifact. Sao lưu thành công chưa đồng nghĩa validation đã đạt.


In [ ]:
from datetime import datetime, timezone
import json
import shutil
import tempfile
import uuid
import zipfile


class IngestionBackup:
    """Save closed ingestion artifacts; run after prepare_datasets() finishes."""

    @staticmethod
    def save(pipeline, destination, *, splits):
        from app.local_pipeline import DatasetPreparationPipeline

        out = Path(pipeline.config.out_dir).resolve()
        dataset = Path(pipeline.config.dataset_dir).resolve()
        destination = Path(destination).resolve()
        if destination == out or out in destination.parents:
            raise ValueError("Thư mục backup phải nằm ngoài OUT")
        splits = tuple(splits)
        if not splits or any(s not in ("train", "public_test") for s in splits):
            raise ValueError("Invalid backup splits")
        groups, businesses = [], []
        for split in splits:
            manifest_dir = out / "manifests" / split
            for business in DatasetPreparationPipeline.businesses(dataset / split):
                current = out / "databases" / split / business.name / "current"
                required = [current / "data.duckdb", current / "catalog.json",
                            manifest_dir / f"{business.name}.json"]
                for artifact in required:
                    if not artifact.is_file():
                        raise FileNotFoundError(f"Chưa ingestion xong: {artifact}")
                groups.append((f"{split}/{business.name}", current))
                businesses.append({"split": split, "business_id": business.name,
                                   "database": (current / "data.duckdb").relative_to(out).as_posix()})
            shared = out / "databases" / split / "_shared" / "current"
            for name in ("data.duckdb", "data.source.json"):
                if not (shared / name).is_file():
                    raise FileNotFoundError(shared / name)
            groups.extend([(f"{split}/calendar", shared), (f"{split}/manifests", manifest_dir)])
        for split in splits:
            report = out / "reports" / f"prepare_{split}.json"
            if report.is_file():
                groups.append((f"reports/{split}", report))

        # Preflight before creating a backup; a WAL may mean a database is still open.
        entries = []
        for label, directory in groups:
            paths = sorted(directory.rglob("*")) if directory.is_dir() else [directory]
            if any(p.is_symlink() or p.suffix == ".wal" for p in paths):
                raise RuntimeError(f"Có symlink hoặc DuckDB WAL tại {directory}; đóng kết nối trước khi lưu")
            entries.append((label, [p for p in paths if p.is_file()]))
        created = datetime.now(timezone.utc)
        name = f"ingestion_{'-'.join(splits)}_{created:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}.zip"
        index = {"format_version": 1, "created_at": created.isoformat(),
                 "dataset_dir": str(dataset), "model": pipeline.config.model,
                 "businesses": businesses, "splits": list(splits),
                 "note": "Ingestion snapshot; run validate_databases() after restoring."}
        destination.mkdir(parents=True, exist_ok=True)
        target = destination / name
        partial = destination / (name + ".partial")
        # Build on the Colab disk, then copy a single file to Drive.
        with tempfile.TemporaryDirectory(prefix="datamate_backup_") as temporary:
            archive = Path(temporary) / name
            with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1,
                                 allowZip64=True) as bundle:
                for number, (label, paths) in enumerate(entries, 1):
                    print(f"[backup {number}/{len(entries)}] {label}: {len(paths)} files", flush=True)
                    for source in paths:
                        bundle.write(source, source.relative_to(out).as_posix())
                bundle.writestr("backup_index.json", json.dumps(index, ensure_ascii=False, indent=2))
            print(f"Đang lưu {archive.stat().st_size / 1024**2:.1f} MiB → {target}", flush=True)
            try:
                shutil.copyfile(archive, partial)
                partial.replace(target)
            finally:
                partial.unlink(missing_ok=True)
        print(f"Đã lưu {len(businesses)} business + {len(splits)} calendar: {target}", flush=True)
        return target


# Drive chỉ dùng để lưu backup; dataset vẫn lấy từ repo, DuckDB chạy tại OUT.
BACKUP_DIR = "/content/drive/MyDrive/datamate_ingestion_backups"
runtime.prepare_output(BACKUP_DIR)
backup_path = IngestionBackup.save(pipeline, BACKUP_DIR, splits=SPLITS)

# Nếu muốn tải bản ZIP về máy:
# from google.colab import files
# files.download(str(backup_path))


### 3.2. Kiểm tra dữ liệu đã lưu
Bước này kiểm tra fingerprint nguồn, catalog và khả năng truy vấn bảng; không gọi Qwen. Tách riêng để dễ phân biệt thời gian ingestion, sao lưu và validation. Chỉ tiếp tục smoke test khi kiểm tra thành công.


In [ ]:
from time import perf_counter

validation_started = perf_counter()
print(f"Bắt đầu kiểm tra DuckDB của {DATASET_SPLIT}...", flush=True)
validation = pipeline.validate_databases(splits=SPLITS)
print(f"Validation hoàn tất sau {perf_counter() - validation_started:.1f}s", flush=True)


## 4. Smoke test: 3 câu
Agent chỉ dùng database đã chuẩn bị và còn hợp lệ. Qwen chuẩn hóa câu hỏi, sinh JSON chứa SQL, nhận lỗi DuckDB để sửa và áp dụng đồng thuận. Xem lỗi sinh/SQL và VRAM trước khi chạy toàn bộ train.
Chỉ chạy khi `DATASET_SPLIT == "train"`.


In [ ]:
if DATASET_SPLIT == "train":
    _, _, smoke_scores, _, smoke_logs = pipeline.smoke_test()
    for row in smoke_logs:
        print(row.get("id"), row.get("type"), row.get("sql"), row.get("errors"))
else:
    print("public_test: tiếp tục đến bước dự đoán ở mục 6.")


## 5. Đánh giá toàn bộ train
Mặc định xử lý tất cả câu train, không giới hạn thời gian. Đáp án chuẩn chỉ dùng sau suy luận để chấm PA. Khi cần đo holdout: dùng `pipeline.evaluate(subset="holdout", holdout_n=6, per_biz=5)` thay lời gọi phía dưới. Thời gian toàn bộ train cần ước tính từ smoke test; chưa có số đo GPU thực tế trong notebook.
Chỉ chạy khi `DATASET_SPLIT == "train"`.


In [ ]:
if DATASET_SPLIT == "train":
    _, _, train_scores, _, train_logs = pipeline.evaluate()
    print(train_scores.status.value_counts())
else:
    print("public_test không có đáp án chuẩn; chạy dự đoán ở mục 6.")


## 6. Dự đoán toàn bộ public test và kiểm định bài nộp
Không giới hạn số câu/thời gian. Kết quả ở `OUT/public_test/submission.jsonl`, log kèm SQL và lỗi từng câu. Dùng lại cùng đối tượng `pipeline` nên không tải lại Qwen. Trước khi reset runtime, tải file hoặc sao lưu `OUT` nếu đang dùng ổ `/content`.
Tự động chạy khi `DATASET_SPLIT == "public_test"`; bỏ qua khi chọn train.


In [ ]:
if DATASET_SPLIT == "public_test":
    submission_path = pipeline.predict_public_test()
    print("Bài nộp:", submission_path)
else:
    print("Đã chọn train; kết quả đánh giá nằm trong OUT/train.")

# Tải bài nộp về máy khi DATASET_SPLIT == "public_test":
# from google.colab import files
# files.download(str(submission_path))
